# Notebook 01 — Prepare DAIC-WOZ & E-DAIC (real data → features + labels)

**Run this FIRST, on your own PC** (the data stays on your local drive, nothing is uploaded).

It turns your two datasets into **subject-level feature tables** the modelling notebooks can
use, straight from the facial-expression features they already contain:

- **DAIC-WOZ** (`/path/to/DAIC-WOZ`) — CLNF Action Units, 2D/3D landmarks, gaze, head pose.
- **E-DAIC** (`/path/to/E-DAIC`) — OpenFace 2.1 AUs / gaze / pose (auto-discovered).

### Important — these datasets have NO raw face images
They ship *pre-extracted* facial-expression features, not video frames. So:
- They support facial-cue analysis and benchmark evaluation, which is what this notebook prepares.
- A pixel-based CNN/ViT study needs raw frames (e.g. AVEC 2013/2014).

### What it outputs (to `outputs/prep/`)
- `daic_subject_features.csv`, `edaic_subject_features.csv` — one row per participant:
  `pid, split, gender, phq_score, phq_binary, <hundreds of expression features>`.
- A discovery report so you can see precisely which files were found in each dataset.
- Baseline results (binary + severity) proving the data flows into a model.

Only `numpy / pandas / scikit-learn` are needed — no GPU, no PyTorch.

## 1. Config — edit the two paths if yours differ, then Run All

In [ ]:
import os, re, glob, warnings, json
import numpy as np, pandas as pd
warnings.filterwarnings("ignore")
try:
    from tqdm.auto import tqdm
except Exception:
    def tqdm(x, **k): return x

DAIC_ROOT  = r"/path/to/DAIC-WOZ"
EDAIC_ROOT = r"/path/to/E-DAIC"

OUT_DIR           = "outputs/prep"
CONF_THRESHOLD    = 0.0      # OpenFace/CLNF confidence 0..1; e.g. 0.90 to drop low-confidence frames
INCLUDE_LANDMARKS = False    # True adds 68-point landmark coords (many more features)
PERCENTILES       = [25, 50, 75]
os.makedirs(OUT_DIR, exist_ok=True)
print("DAIC :", DAIC_ROOT, "| exists:", os.path.isdir(DAIC_ROOT))
print("E-DAIC:", EDAIC_ROOT, "| exists:", os.path.isdir(EDAIC_ROOT))

## 2. Discovery + parsing functions

Everything is driven by **column names**, so the same code handles DAIC's CLNF files and
E-DAIC's OpenFace file without hard-coding names. Meta columns (`frame`, `timestamp`,
`confidence`, `success`) are used only for filtering; audio/text/HOG/CNN files are skipped.

In [ ]:
META = {"frame", "face_id", "timestamp", "confidence", "success", "timer"}
AU_R = re.compile(r"^AU\d+_r$"); AU_C = re.compile(r"^AU\d+_c$")
SKIP = ("hog", "covarep", "formant", "transcript", "audio", "boaw", "bovw", "egemaps",
        "mfcc", "densenet", "resnet", "vgg", "deepspectrum", "bert", "mel")

def participant_dirs(root):
    out = []
    if not os.path.isdir(root): return out
    for dp, dn, fn in os.walk(root):
        if re.fullmatch(r"\d{3,4}_P", os.path.basename(dp)):
            out.append(dp)
            dn[:] = []          # don't descend: E-DAIC has a nested NNN_P inside NNN_P
    return sorted(out)

def pid_of(pdir):
    return int(re.match(r"(\d+)_P", os.path.basename(pdir)).group(1))

def _headers(path):
    try:
        return [c.strip() for c in pd.read_csv(path, nrows=0, skipinitialspace=True).columns]
    except Exception:
        return []

def feature_files(pdir):
    files = []
    for f in glob.glob(os.path.join(pdir, "**", "*.*"), recursive=True):
        name = os.path.basename(f).lower()
        if name.rsplit(".", 1)[-1] not in ("txt", "csv"): continue
        if any(s in name for s in SKIP): continue
        files.append(f)
    return files

def classify(path, cols):
    name = os.path.basename(path).lower()
    has_au = any(AU_R.match(c) or AU_C.match(c) for c in cols)
    if has_au:                                  # DAIC AUs file OR E-DAIC combined OpenFace
        return "au"
    if "feature" in name:                       # CLNF landmark files (features / features3D)
        return "landmark"
    if "gaze" in name or any(c.startswith("gaze") for c in cols):
        return "gaze"
    if ("pose" in name or any(c in ("Tx","Ty","Tz","Rx","Ry","Rz") for c in cols)
            or any(c.startswith("pose_") for c in cols)):
        return "pose"
    return "other"

def load_block(path, kind):
    df = pd.read_csv(path, skipinitialspace=True)
    df.columns = [c.strip() for c in df.columns]
    if "success" in df.columns:  df = df[df["success"] == 1]
    if CONF_THRESHOLD > 0 and "confidence" in df.columns:
        df = df[df["confidence"] >= CONF_THRESHOLD]
    cols = [c for c in df.columns if c not in META and not c.startswith("Unnamed")]
    if kind == "au":                            # keep AUs (+ gaze/pose if a combined OpenFace file)
        sel = [c for c in cols if AU_R.match(c) or AU_C.match(c)
               or c.startswith("gaze_") or c.startswith("pose_R") or c.startswith("pose_T")]
        cols = sel or cols
    return df[cols].apply(pd.to_numeric, errors="coerce")

def functionals(num, prefix):
    out = {}
    for c in num.columns:
        col = num[c].dropna()
        if col.empty: continue
        out[f"{prefix}:{c}:mean"] = col.mean()
        out[f"{prefix}:{c}:std"]  = col.std()
        out[f"{prefix}:{c}:min"]  = col.min()
        out[f"{prefix}:{c}:max"]  = col.max()
        for p in PERCENTILES:
            out[f"{prefix}:{c}:p{p}"] = col.quantile(p / 100.0)
    return out

def participant_features(pdir, report=None):
    feats = {}
    for f in feature_files(pdir):
        cols = _headers(f)
        if not cols: continue
        kind = classify(f, cols)
        if kind == "other": continue
        if kind == "landmark" and not INCLUDE_LANDMARKS: continue
        if report is not None:
            report.append(f"{os.path.basename(f)}  ->  {kind}  ({len(cols)} cols)")
        try:
            feats.update(functionals(load_block(f, kind), kind))
        except Exception as e:
            if report is not None: report.append(f"   (skip {os.path.basename(f)}: {e})")
    return feats

print("parsing functions defined.")

## 3. Discovery report — see what's really inside each dataset

Prints the participant count and, for the first participant of each dataset, which files were
found and how they were classified. This is where E-DAIC's `data\` layout reveals itself.

In [ ]:
for tag, root in [("DAIC", DAIC_ROOT), ("E-DAIC", EDAIC_ROOT)]:
    print("=" * 64); print(tag, "->", root)
    pdirs = participant_dirs(root)
    pids = [pid_of(d) for d in pdirs]
    print("participant folders:", len(pdirs), "| unique IDs:", len(set(pids)))
    if pdirs:
        rep = []
        _ = participant_features(pdirs[0], report=rep)
        print("sample:", os.path.basename(pdirs[0]))
        for line in rep: print("   ", line)
    else:
        print("   (no NNN_P folders found — check the path)")

## 4. Labels from the official splits

In [ ]:
def load_daic_labels(root):
    frames = []
    for split in ["train", "dev", "test"]:
        g = glob.glob(os.path.join(root, f"{split}_split*.csv"))
        if not g: continue
        d = pd.read_csv(g[0]); d.columns = [c.strip() for c in d.columns]
        d = d.rename(columns={"Participant_ID": "pid", "participant_ID": "pid",
                              "PHQ8_Binary": "phq_binary", "PHQ8_Score": "phq_score",
                              "Gender": "gender"})
        for col in ["phq_binary", "phq_score"]:
            if col not in d: d[col] = np.nan
        d["split"] = split
        frames.append(d[["pid", "phq_binary", "phq_score", "gender", "split"]])
    if not frames:
        print("   (no DAIC split CSVs found)"); return pd.DataFrame()
    lab = pd.concat(frames, ignore_index=True)
    lab["gender"] = lab["gender"].map({1: "F", 0: "M"}).fillna(lab["gender"].astype(str))
    return lab

def load_edaic_labels(root):
    base = os.path.join(root, "labels2019", "labels")
    if not os.path.isdir(base): base = os.path.join(root, "labels")
    frames = []
    for split in ["train", "dev", "test"]:
        p = os.path.join(base, f"{split}_split.csv")
        if not os.path.exists(p): continue
        d = pd.read_csv(p); d.columns = [c.strip() for c in d.columns]
        d = d.rename(columns={"Participant_ID": "pid", "PHQ_Binary": "phq_binary",
                              "PHQ_Score": "phq_score", "Gender": "gender"})
        for col in ["phq_binary", "phq_score"]:
            if col not in d: d[col] = np.nan
        d["split"] = split
        frames.append(d[["pid", "phq_binary", "phq_score", "gender", "split"]])
    if not frames:
        print("   (no E-DAIC split CSVs found)"); return pd.DataFrame()
    return pd.concat(frames, ignore_index=True)

daic_labels  = load_daic_labels(DAIC_ROOT)   if os.path.isdir(DAIC_ROOT)  else pd.DataFrame()
edaic_labels = load_edaic_labels(EDAIC_ROOT) if os.path.isdir(EDAIC_ROOT) else pd.DataFrame()
for tag, lab in [("DAIC", daic_labels), ("E-DAIC", edaic_labels)]:
    if len(lab):
        print(f"{tag}: {len(lab)} labelled participants | splits:",
              dict(lab.split.value_counts()),
              "| depressed:", int(lab.phq_binary.sum(skipna=True)) if lab.phq_binary.notna().any() else "n/a")

## 5. Build subject-level feature tables

For every participant we load each expression file, filter to tracked frames, and summarise
each feature over the whole session (mean / std / min / max / percentiles). One row per
participant, joined to its label. (Hundreds of participants × tens of thousands of frames —
this cell takes a few minutes the first time.)

In [ ]:
def build_dataset(root, labels):
    rows = []
    for pdir in tqdm(participant_dirs(root), desc=os.path.basename(root)):
        feats = participant_features(pdir)
        if not feats: continue
        feats["pid"] = pid_of(pdir)
        rows.append(feats)
    if not rows: return pd.DataFrame()
    F = pd.DataFrame(rows).set_index("pid")
    if F.index.duplicated().any():
        print(f"   WARNING: {F.index.duplicated().sum()} duplicate participants dropped")
        F = F[~F.index.duplicated(keep="first")]
    lab = labels.drop_duplicates("pid").set_index("pid")
    df = lab.join(F, how="inner").reset_index()
    assert df["pid"].is_unique, "LEAKAGE GUARD: a participant appears more than once"
    return df

daic = build_dataset(DAIC_ROOT, daic_labels)   if len(daic_labels)  else pd.DataFrame()
edaic = build_dataset(EDAIC_ROOT, edaic_labels) if len(edaic_labels) else pd.DataFrame()

for tag, df, fn in [("DAIC", daic, "daic_subject_features.csv"),
                    ("E-DAIC", edaic, "edaic_subject_features.csv")]:
    if len(df):
        nfeat = df.shape[1] - 5
        df.to_csv(os.path.join(OUT_DIR, fn), index=False)
        print(f"{tag}: {df.shape[0]} participants x {nfeat} features -> {OUT_DIR}/{fn}")
    else:
        print(f"{tag}: no features built (check discovery report in section 3)")

## 6. Baseline sanity check (proves data → model works)

Binary depression (macro-F1 / balanced-acc / AUC) and severity (RMSE / MAE / CCC). DAIC uses
subject-independent 5-fold CV on train+dev (official test has no labels); E-DAIC uses the
official train→test split. These are honest starting numbers, not tuned.

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, balanced_accuracy_score, roc_auc_score, mean_squared_error, mean_absolute_error

def ccc(y, p):
    y, p = np.asarray(y, float), np.asarray(p, float)
    vy, vp = y.var(), p.var()
    if vy == 0 and vp == 0: return 1.0
    cov = np.mean((y - y.mean()) * (p - p.mean()))
    return float(2 * cov / (vy + vp + (y.mean() - p.mean()) ** 2 + 1e-12))

def Xy(df):
    feat = [c for c in df.columns if c not in ("pid","split","gender","phq_binary","phq_score")]
    X = SimpleImputer(strategy="median").fit_transform(df[feat])
    return X, df

def _safe_k(y, cap=5):
    _, counts = np.unique(np.asarray(y), return_counts=True)
    return max(2, min(cap, int(counts.min())))

def cv_binary(df):
    X, d = Xy(df); y = d.phq_binary.values.astype(int)
    if len(set(y)) < 2 or min(np.bincount(y)) < 2:
        print("   binary: too few samples in a class — skipping CV"); return
    k = _safe_k(y)
    skf = StratifiedKFold(k, shuffle=True, random_state=42); f1=[]; ba=[]; au=[]
    clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, class_weight="balanced"))
    for tr, te in skf.split(X, y):
        clf.fit(X[tr], y[tr]); pr = clf.predict(X[te]); pp = clf.predict_proba(X[te])[:,1]
        f1.append(f1_score(y[te], pr, average="macro")); ba.append(balanced_accuracy_score(y[te], pr))
        au.append(roc_auc_score(y[te], pp) if len(set(y[te]))>1 else np.nan)
    print(f"   binary ({k}-fold): macro-F1={np.mean(f1):.3f}  bal-acc={np.mean(ba):.3f}  AUC={np.nanmean(au):.3f}")

def cv_sev(df):
    d = df[df.phq_score.notna()]
    if len(d) < 8: print("   severity: too few labelled scores — skipping"); return
    X, d = Xy(d); y = d.phq_score.values.astype(float)
    k = max(2, min(5, len(y)//2))
    strat = pd.qcut(y, min(4, len(np.unique(y))), labels=False, duplicates="drop")
    skf = StratifiedKFold(k, shuffle=True, random_state=42)
    rm=[]; ma=[]; cc=[]
    for tr, te in skf.split(X, strat):
        rg = make_pipeline(StandardScaler(), Ridge(alpha=10.0)).fit(X[tr], y[tr]); pr = rg.predict(X[te])
        rm.append(mean_squared_error(y[te], pr)**0.5); ma.append(mean_absolute_error(y[te], pr)); cc.append(ccc(y[te], pr))
    print(f"   severity ({k}-fold): RMSE={np.mean(rm):.2f}  MAE={np.mean(ma):.2f}  CCC={np.mean(cc):.3f}")

def official_split(df):
    tr = df[df.split.isin(["train","dev"]) & df.phq_binary.notna()]
    te = df[(df.split=="test") & df.phq_binary.notna()]
    if len(te)==0: print("   (no labelled test split)"); return
    feat=[c for c in df.columns if c not in ("pid","split","gender","phq_binary","phq_score")]
    imp=SimpleImputer(strategy="median").fit(tr[feat])
    clf=make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, class_weight="balanced"))
    clf.fit(imp.transform(tr[feat]), tr.phq_binary.astype(int))
    Xte=imp.transform(te[feat]); yte=te.phq_binary.astype(int)
    pr=clf.predict(Xte); pp=clf.predict_proba(Xte)[:,1]
    print(f"   official train->test: macro-F1={f1_score(yte,pr,average='macro'):.3f}  "
          f"bal-acc={balanced_accuracy_score(yte,pr):.3f}  "
          f"AUC={roc_auc_score(yte,pp) if len(set(yte))>1 else float('nan'):.3f}")

if len(daic):
    print("DAIC-WOZ (train+dev, subject-independent CV):"); cv_binary(daic[daic.phq_binary.notna()]); cv_sev(daic)
if len(edaic):
    print("E-DAIC (5-fold CV):"); cv_binary(edaic[edaic.phq_binary.notna()]); cv_sev(edaic)
    print("E-DAIC (official split):"); official_split(edaic)